In [1]:
import pandas as pd
import os
import pickle


In [2]:
def main(path_save, file_stem):
    """
    Compute summary statistics of component areas from final masks.

    This function:
    1. Loads the corrected final component masks from a saved bundle.
    2. Converts pixel counts of each component into physical area (mm²).
    3. Computes mean and standard deviation of component areas.
    4. Returns summary statistics for downstream analysis.

    Parameters
    ----------
    path_save : str
        Directory containing the saved component bundle.
    file_stem : str
        Base filename used to locate the bundle.

    Returns
    -------
    dict
        Dictionary containing:
        - 'area_mean' : float
            Mean component area (mm²).
        - 'area_sd' : float
            Standard deviation of component areas (mm²).
        - 'n_component' : int
            Number of components analyzed.

    Notes
    -----
    - Assumes square pixels with spatial resolution of 6.5 µm per pixel.
    - Component area is computed as the number of active pixels x pixel area.
    - Pixel area is converted from µm² to mm².
    """
    #Parameters
    sp_res=6.5 # pixel size in um

    pixel_area = (sp_res * 1e-3)**2 # pixel area in sq mm
    file_data =os.path.join(path_save, file_stem +"_final_components.pkl")
    with open(file_data, "rb") as f:
        bundle = pickle.load(f)
    final_components=bundle["final_components_corrected"]
    area=final_components.sum(axis=1).sum(axis=1)*pixel_area
    return {"area_mean": area.mean(), "area_sd": area.std(), "n_component": len(area)}

 

In [3]:
data_source = {
"exp1_21_06_22_AL1322_P0pups_Gad-KCC2-KO":["pup2_1_spont"]
}

In [4]:
path_dist = "Data"
genotype=pd.read_excel(os.path.join(path_dist, "genotypes_exp1_12.xlsx"), index_col=0)

pup_list=[]
area_mean=[]
area_sd=[]
n_component=[]
for folder_exp in data_source.keys():
    pups=data_source[folder_exp]
    for file_stem in pups:
        os.makedirs(os.path.join(path_dist, "NMF_analysis_results", folder_exp, file_stem), exist_ok=True)
        path_save=os.path.join(path_dist, "NMF_analysis_results", folder_exp, file_stem)
        pup = folder_exp.split("_")[0] + "_" + file_stem.split("_")[0]

        data = main(path_save,file_stem)
        pup_list.append(pup)
        area_mean.append(data["area_mean"])
        area_sd.append(data["area_sd"])
        n_component.append(data["n_component"])
        print(f"Processing {file_stem} from {folder_exp}")
out=pd.DataFrame(index=pup_list, data={"area_mean": area_mean, "area_sd": area_sd,"n_component":n_component})
out["genotype"]=genotype["genotype"]
os.makedirs(os.path.join(path_dist,"NMF_analysis_results"), exist_ok=True)
out.to_excel(os.path.join(path_dist, "NMF_analysis_results","nmf_compounds_area_statistics.xlsx"))


Processing pup2_1_spont from exp1_21_06_22_AL1322_P0pups_Gad-KCC2-KO
